# Training Loop Validation

Run a short training/validation pass to validate the training loop and metrics.

In [5]:
import sys
from pathlib import Path

candidate_roots = [
    Path('/content/ouroboros'),
]
project_root = next((p for p in candidate_roots if p.exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import Subset, DataLoader

from src.data_loaders import get_cifar10_loaders
from src.models import CNN3Layer
from src.trainer import train_epoch, validate_epoch
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()

train_loader, val_loader = get_cifar10_loaders(batch_size=128, num_workers=2, data_dir='assets')

# Create small subsets for a quick validation run
train_subset = Subset(train_loader.dataset, range(1024))
val_subset = Subset(val_loader.dataset, range(512))
train_subset_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
val_subset_loader = DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=2)

model = CNN3Layer(num_classes=10, in_channels=3).to(device)
optimizer = Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_metrics = train_epoch(
    model=model,
    dataloader=train_subset_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    amp_enabled=(device.type == 'cuda'),
    use_compile=False,
    collect_grad_stats=True,
    collect_timing=True,
)

val_metrics = validate_epoch(
    model=model,
    dataloader=val_subset_loader,
    criterion=criterion,
    device=device,
)

print('Train metrics:', train_metrics)
print('Val metrics:', val_metrics)

/content/ouroboros/src/trainer.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_to_use = scaler if (scaler is not None) else (torch.cuda.amp.GradScaler() if amp_active else None)
/content/ouroboros/src/trainer.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():


Train metrics: {'loss': 2.188809394836426, 'accuracy': 0.1875, 'gradients': {'total_l2_norm': 0.6170585479067147, 'per_layer_l2_norms': {'conv1.weight': 0.26838743686676025, 'conv1.bias': 2.9986097160872305e-06, 'bn1.weight': 0.011946653015911579, 'bn1.bias': 0.012256997637450695, 'conv2.weight': 0.27587297558784485, 'conv2.bias': 2.267777517772629e-06, 'bn2.weight': 0.011774447746574879, 'bn2.bias': 0.008318723179399967, 'conv3.weight': 0.3139481842517853, 'conv3.bias': 1.6772177104940056e-06, 'bn3.weight': 0.024319220334291458, 'bn3.bias': 0.022765645757317543, 'fc.weight': 0.3595307767391205, 'fc.bias': 0.05645529553294182}, 'zero_grad_parameters': 0}, 'time_sec': 2.1304009410000617, 'samples_per_sec': 480.6606964411636}
Val metrics: {'loss': 2.2284274101257324, 'accuracy': 0.1953125}
